In [1]:
import os

import pandas as pd
import requests
from bs4 import BeautifulSoup

from toronto_bylaw_parser import extract_content_rows, extract_table_of_contents

In [2]:
base_url = "https://www.toronto.ca/zoning/bylaw_amendments/"
root_page = "ZBL_NewProvision_Chapter1.htm"
cache_path = "data/toronto/input/ZBL_NewProvision_Chapter1.htm"

if os.path.exists(cache_path):
    print("file exists")
    with open(cache_path, encoding="utf-8") as file:
        html = file.read()
else:
    print("fetching file")
    response = requests.get(base_url + root_page)
    html = response.text
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, "w", encoding="utf-8") as file:
        file.write(html)

soup = BeautifulSoup(html, "html.parser")

file exists


In [3]:
tables = soup.find_all("table")
toc_index = next((i for i, x in enumerate(tables) if "Table of Contents" in x.text), None)

if toc_index is not None:
    toc = tables[toc_index]
    # left column - table of contents
    toc_data = extract_table_of_contents(toc)

else:
    print("No Table of Contents found")

In [ ]:
docs = []


for doc_num, doc in enumerate(toc_data):
    doc_link = doc["link"]
    cache_path = f"data/toronto/input/{doc_link}"
    print(cache_path)

    if os.path.exists(cache_path):
        print("file exists")
        with open(cache_path, encoding="utf-8") as file:
            html = file.read()
    else:
        print("fetching file")
        response = requests.get(base_url + doc_link)
        html = response.text
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, "w", encoding="utf-8") as file:
            file.write(html)

    soup = BeautifulSoup(html, "html.parser")

    content_header = soup.find("h1")
    if content_header is not None and content_header.parent is not None:
        content_section = content_header.parent
        page_content = extract_content_rows(content_section)
        for item in page_content:
            item.update(
                {
                    "doc_id": doc_num,
                    "link": doc_link,
                    "toc_section": doc["section"],
                    "toc_title": doc["title"],
                }
            )
        docs.extend(page_content)

data/toronto/input/ZBL_NewProvision_Chapter1.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter1.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter2.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter5.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter10.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter15.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter30.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter40.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter50.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter60.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter80.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter90.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter100.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter150.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter200.htm
file exists
data/toronto/input/ZBL_NewProvision_Chapter220.htm
file exists
data/tor

In [5]:
pd.DataFrame(docs).to_csv("data/toronto/output/bylaw_processed.csv", index=False)